# Week 5 Day 5: Mini Project - Student Performance Prediction

Choose a dataset, define the prediction problem, prepare features, train and test at least two models, compare their performance metrics, and select the optimal model.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

# 1. Generate Synthetic Student Performance Dataset
np.random.seed(42)
n_samples = 400

data = {
    "Study_Hours": np.random.uniform(1, 15, size=n_samples),
    "Attendance_Rate": np.random.uniform(50, 100, size=n_samples),
    "Previous_Score": np.random.uniform(50, 98, size=n_samples),
    "Extracurricular": np.random.choice(["Yes", "No"], size=n_samples),
    "Passed": np.random.choice([0, 1], size=n_samples, p=[0.3, 0.7]),
}

df = pd.DataFrame(data)

# Introduce a few missing values to test preprocessing pipeline
df.loc[10:15, "Study_Hours"] = np.nan
df.loc[20:22, "Previous_Score"] = np.nan

X = df.drop(columns=["Passed"])
y = df["Passed"]

# 2. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Setup Preprocessing Pipeline
num_features = ["Study_Hours", "Attendance_Rate", "Previous_Score"]
cat_features = ["Extracurricular"]

num_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

cat_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(drop="first")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_transformer, num_features),
        ("cat", cat_transformer, cat_features),
    ]
)

# 4. Train and Benchmark Two Models
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
}

results = []

for name, model in models.items():
  pipeline = Pipeline(
      steps=[("preprocessor", preprocessor), ("classifier", model)]
  )
  pipeline.fit(X_train, y_train)
  preds = pipeline.predict(X_test)

  acc = accuracy_score(y_test, preds)
  f1 = f1_score(y_test, preds)

  results.append({"Model": name, "Accuracy": acc, "F1 Score": f1})
  print(f"--- {name} Evaluation ---")
  print(classification_report(y_test, preds))

# 5. Summary Comparison
results_df = pd.DataFrame(results)
print("\n--- Model Comparison Summary ---")
print(results_df.to_string(index=False))